In [27]:
import pandas as pd
import sys
from datetime import datetime

# 1. 📂 Configuration des fichiers
input_path = "/Users/titou/Desktop/PT/consolidation-etalab-schema-irve-statique-v-2.3.1-20250322.csv"
output_report_path = "/Users/titou/Desktop/PT/rapport_bornes_irve.txt"

# 2. 📝 Préparation de la capture du rapport
class ReportCapture:
    def __init__(self):
        self.content = []
    
    def write(self, text):
        self.content.append(text)
    
    def get_content(self):
        return "".join(self.content)

# Redirection de la sortie standard
original_stdout = sys.stdout
report_capture = ReportCapture()
sys.stdout = report_capture

# 3. 🛠 Traitement des données
try:
    print(f"=== RAPPORT D'ANALYSE IRVE - {datetime.now().strftime('%d/%m/%Y %H:%M')} ===\n")
    
    # Lecture du fichier avec détection automatique du séparateur
    for sep in [',', ';', '\t']:
        try:
            df = pd.read_csv(input_path, sep=sep, encoding='utf-8', low_memory=False)
            if len(df.columns) > 1:
                break
        except:
            continue
    
    # Nettoyage des colonnes
    df.columns = df.columns.str.lower().str.replace("[^a-z0-9_]", "_", regex=True)
    
    print(f"✅ Fichier chargé avec succès ({len(df)} lignes)")
    print("\n🔍 COLONNES DISPONIBLES :")
    print(df.columns.tolist())
    
    # Analyse des opérateurs
    if 'nom_operateur' in df.columns:
        operateurs = df['nom_operateur'].value_counts(dropna=False)
        print("\n🔌 TOP 20 DES OPÉRATEURS :")
        print(operateurs.head(20))
    else:
        print("\n⚠️ Colonne 'nom_operateur' introuvable")
    
    # Bornes Tesla
    if 'nom_operateur' in df.columns:
        tesla_mask = df['nom_operateur'].str.contains('tesla', case=False, na=False)
        bornes_tesla = df[tesla_mask]
        print(f"\n⚡ NOMBRE DE BORNES TESLA : {len(bornes_tesla)}")
        if not bornes_tesla.empty:
            print(bornes_tesla[['nom_operateur', 'nom_station', 'adresse_station']].head(10))
    else:
        print("\n⚠️ Impossible d'analyser les bornes Tesla (colonne manquante)")
    
    # Bornes privées
    if 'condition_acces' in df.columns:
        prive_mask = df['condition_acces'].str.contains('privé|restreint|réservé', case=False, na=False, regex=True)
        bornes_privees = df[prive_mask]
        print(f"\n🔒 NOMBRE DE BORNES PRIVÉES : {len(bornes_privees)}")
        if not bornes_privees.empty:
            print(bornes_privees[['nom_operateur', 'condition_acces', 'nom_station']].head(10))
    else:
        print("\n⚠️ Impossible d'analyser les bornes privées (colonne manquante)")

except Exception as e:
    print(f"\n❌ ERREUR : {str(e)}")

finally:
    # 4. 💾 Sauvegarde du rapport uniquement
    sys.stdout = original_stdout
    
    with open(output_report_path, 'w', encoding='utf-8') as f:
        f.write(report_capture.get_content())
    
    print(f"\n📄 Rapport texte sauvegardé : {output_report_path}")
    print("="*60)
    print(report_capture.get_content())  # Affiche le rapport dans la console


📄 Rapport texte sauvegardé : /Users/titou/Desktop/PT/rapport_bornes_irve.txt
=== RAPPORT D'ANALYSE IRVE - 16/04/2025 18:59 ===

✅ Fichier chargé avec succès (127260 lignes)

🔍 COLONNES DISPONIBLES :
['nom_amenageur', 'siren_amenageur', 'contact_amenageur', 'nom_operateur', 'contact_operateur', 'telephone_operateur', 'nom_enseigne', 'id_station_itinerance', 'id_station_local', 'nom_station', 'implantation_station', 'adresse_station', 'code_insee_commune', 'coordonneesxy', 'nbre_pdc', 'id_pdc_itinerance', 'id_pdc_local', 'puissance_nominale', 'prise_type_ef', 'prise_type_2', 'prise_type_combo_ccs', 'prise_type_chademo', 'prise_type_autre', 'gratuit', 'paiement_acte', 'paiement_cb', 'paiement_autre', 'tarification', 'condition_acces', 'reservation', 'horaires', 'accessibilite_pmr', 'restriction_gabarit', 'station_deux_roues', 'raccordement', 'num_pdl', 'date_mise_en_service', 'observations', 'date_maj', 'cable_t2_attache', 'last_modified', 'datagouv_dataset_id', 'datagouv_resource_id', '